# Phase 2: Knowledge Graph Extraction (Standard LangChain & Neo4j)

이 노트북에서는 Phase 1에서 파싱한 JSON 청크 데이터를 불러와, `langchain_experimental`의 `LLMGraphTransformer`를 사용하여 텍스트에서 Entity와 Relation을 추출합니다. 이후 추출된 지식망을 로컬 또는 클라우드의 **Neo4j 그래프 데이터베이스**에 적재(Insert)합니다.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community langchain-experimental neo4j python-dotenv

## 1. 환경 변수 세팅
이 코드를 실행하려면 프로젝트 루트(또는 현재 폴더)에 `.env` 파일이 있어야 하며, 다음 값들이 설정되어 있어야 합니다.
- `OPENAI_API_KEY`
- `NEO4J_URI`
- `NEO4J_USERNAME`
- `NEO4J_PASSWORD`

In [ ]:
import os
import sys
from dotenv import load_dotenv

sys.path.append('../../')
load_dotenv('../../.env')

print("Environment Variables Loaded!")

## 2. 데이터 로드 (Step 01 결과물)
Phase 1에서 파싱해 둔 `parsed_chunks.json`을 읽어와 LangChain의 `Document` 객체 리스트로 변환합니다.

In [ ]:
import json
from langchain_core.documents import Document

data_path = "../../graphrag-data-parsing/data/parsed_chunks.json"
with open(data_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

documents = []
for chunk in chunks:
    doc = Document(
        page_content=chunk["content"],
        metadata={"source": chunk["source"], "title": chunk["title"]}
    )
    documents.append(doc)

print(f"Loaded {len(documents)} documents for processing.")

## 3. 지식 그래프 추출 (LLMGraphTransformer)
LangChain이 제공하는 표준 추출 파이프라인을 사용합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm = ChatOpenAI(temperature=0, model_name="gpt-4o-mini")
llm_transformer = LLMGraphTransformer(llm=llm)

print("Extracting graph data using LLM...")
graph_documents = llm_transformer.convert_to_graph_documents(documents)

sample = graph_documents[0]
print(f"\nSample Source: {sample.source.metadata['title']}")
print(f"Nodes Extracted: {len(sample.nodes)}")
print(f"Relationships Extracted: {len(sample.relationships)}")
for rel in sample.relationships[:3]:
    print(f"{rel.source.id} --[{rel.type}]--> {rel.target.id}")

## 4. Neo4j에 그래프 적재 (수동 Cypher 쿼리 방식)
APOC 플러그인 의존성을 제거하고, 오직 순수 Neo4j 드라이버와 표준 Cypher 쿼리(`MERGE`)만을 사용하여 데이터를 매우 빠르고 안정적으로 Insert 합니다.

In [ ]:
from neo4j import GraphDatabase

URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
AUTH = (os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD", "graphrag2026"))

driver = GraphDatabase.driver(URI, auth=AUTH)

def insert_graph_data(tx, graph_docs):
    for doc in graph_docs:
        # 1. 노드(Entity) 생성
        for node in doc.nodes:
            # 동적 라벨 사용 시 Cypher Injection 방지를 위해 알파벳/숫자만 허용하는 것이 좋으나
            # 여기서는 LangChain이 추출한 타입을 그대로 사용합니다.
            label = node.type.replace(" ", "_")
            query = f"MERGE (n:`{label}` {{id: $id}})"
            tx.run(query, id=node.id)
            
        # 2. 엣지(Relationship) 생성
        for rel in doc.relationships:
            src_label = rel.source.type.replace(" ", "_")
            tgt_label = rel.target.type.replace(" ", "_")
            rel_type = rel.type.replace(" ", "_").upper()
            
            query = f"""
            MATCH (source:`{src_label}` {{id: $source_id}})
            MATCH (target:`{tgt_label}` {{id: $target_id}})
            MERGE (source)-[r:`{rel_type}`]->(target)
            """
            tx.run(query, source_id=rel.source.id, target_id=rel.target.id)

print("Loading data into Neo4j database via manual Cypher...")
with driver.session() as session:
    session.execute_write(insert_graph_data, graph_documents)
driver.close()
print("Graph database loading complete! Open Neo4j Browser to view your graph.")